In [4]:
import pandas as pd
import os

downloadable_plots = []

#Nomi dei file CSV
file_o3 = "Statistiche O3 (2002-2022).csv"
file_no2 = "Statistiche NO2 (2001-2022).csv"
file_pm25 = "Statistiche PM 2,5 (2004-2022).csv"
file_pm10 = "Statistiche PM10 (2002-2022).csv"

#Leggi i file CSV in DataFrame pandas
try:
    df_o3 = pd.read_csv(file_o3, delimiter=";")
    df_no2 = pd.read_csv(file_no2, delimiter=";")
    df_pm25 = pd.read_csv(file_pm25, delimiter=";")
    df_pm10 = pd.read_csv(file_pm10, delimiter=";")
except FileNotFoundError:
    print("Uno o più file CSV non sono stati trovati. Assicurati che i nomi dei file siano corretti e che i file si trovino nella stessa directory dello script.")
    exit()

#Funzione per processare un DataFrame e raggruppare per regione e anno
def processa_e_raggruppa_per_regione_anno(df, nome_parametro, colonna_valore):
    if df is None or df.empty: #Aggiungi controllo per df None o vuoto
        print(f"Nessun dato da processare per {nome_parametro}.")
        return None
    
    if "Regione" not in df.columns or "Anno" not in df.columns:
        print(f"Errore: Una o entrambe le colonne 'Regione' e 'Anno' non sono presenti nel file {nome_parametro}.")
        return None

    #Converti le colonne numeriche in numeriche, gestendo errori
    for col in ['Regione', 'Anno', colonna_valore]:
        if col in df.columns:
            if col == 'Anno':
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
            elif col != 'Regione':
                df[col] = pd.to_numeric(df[col], errors='coerce')
                
    #Raggruppa per regione e anno e calcola alcune statistiche
    df_grouped = df.groupby(['Regione', 'Anno']).agg(
        Media=(colonna_valore, 'mean'),
        Minimo=(colonna_valore, 'min'),
        Massimo=(colonna_valore, 'max'),
        DeviazioneStandard=(colonna_valore, 'std'),
        Conteggio=(colonna_valore, 'count')
    )
    return df_grouped

#Processa ogni DataFrame raggruppando per regione e anno
grouped_o3_ra = processa_e_raggruppa_per_regione_anno(df_o3, 'O3', 'giorni_over_240')
grouped_no2_ra = processa_e_raggruppa_per_regione_anno(df_no2, 'NO2', 'sup25d')
grouped_pm25_ra = processa_e_raggruppa_per_regione_anno(df_pm25, 'PM2.5', 'sup25')
grouped_pm10_ra = processa_e_raggruppa_per_regione_anno(df_pm10, 'PM10', 'sup50')

#Visualizza i risultati raggruppati per Regione e Anno
print("\nStatistiche O3 superamento soglia allarme raggruppate per Regione e Anno:")
if grouped_o3_ra is not None:
    print(grouped_o3_ra)

print("\nStatistiche NO2 superamento soglia 25 µg/m^3 raggruppate per Regione e Anno:")
if grouped_no2_ra is not None:
    print(grouped_no2_ra)

print("\nStatistiche PM2.5 superamento soglia 25 µg/m^3 raggruppate per Regione e Anno:")
if grouped_pm25_ra is not None:
    print(grouped_pm25_ra)

print("\nStatistiche PM10 superamento soglia 50 µg/m^3 raggruppate per Regione e Anno:")
if grouped_pm10_ra is not None:
    print(grouped_pm10_ra)

#Salvataggio statistiche raggruppate
if grouped_o3_ra is not None:
    grouped_o3_ra.to_csv('Statistiche O3 Per Regione.csv', sep=';')
if grouped_no2_ra is not None:
    grouped_no2_ra.to_csv('Statistiche NO2 Per Regione.csv', sep=';')
if grouped_pm25_ra is not None:
    grouped_pm25_ra.to_csv('Statistiche PM2,5 Per Regione.csv', sep=';')
if grouped_pm10_ra is not None:
    grouped_pm10_ra.to_csv('Statistiche PM10 Per Regione.csv', sep=';')

#Directory per salvare i grafici
PLOTS_DIR = "Grafici Statistiche"
os.makedirs(PLOTS_DIR, exist_ok=True)


Statistiche O3 superamento soglia allarme raggruppate per Regione e Anno:
              Media  Minimo  Massimo  DeviazioneStandard  Conteggio
Regione Anno                                                       
Abruzzo 2002  0.000       0        0            0.000000          3
        2003  0.000       0        0            0.000000          2
        2004  0.000       0        0            0.000000          5
        2005  0.000       0        0            0.000000          5
        2006  0.000       0        0            0.000000          5
...             ...     ...      ...                 ...        ...
Veneto  2018  0.000       0        0            0.000000         25
        2019  0.375       0        1            0.494535         24
        2020  0.000       0        0            0.000000         24
        2021  0.000       0        0            0.000000         24
        2022  0.000       0        0            0.000000         24

[408 rows x 5 columns]

Statistiche NO2 

In [10]:
import matplotlib.pyplot as plt
import numpy as np

#Dizionario per accedere ai DataFrame raggruppati
grouped_data = {
    'O3': grouped_o3_ra,
    'NO2': grouped_no2_ra,
    'PM2.5': grouped_pm25_ra,
    'PM10': grouped_pm10_ra
}

#Funzione per creare e salvare il grafico a barre
def create_and_save_bar_plot(data, title, y_label, filename):
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ['blue', 'orange', 'green', 'red', 'brown']
    bars = ax.bar(data.index, data.values, color=colors)
    ax.set_ylabel(y_label)
    ax.set_title(title)
    plt.xticks(rotation=45, ha='right', fontsize=9)
    ax.yaxis.grid(True, linestyle='--', alpha=0.6)
    ax.set_facecolor('lightgray')

    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3),
            textcoords="offset points",
            ha='center', va='bottom', fontsize=7)

    plt.tight_layout()
    full_path = os.path.join(PLOTS_DIR, filename)
    fig.savefig(full_path, bbox_inches='tight')
    plt.close(fig)
    return full_path

#Raccogliamo i percorsi dei grafici per la pagina HTML separata
statistiche_per_grafico = ['Media', 'Minimo', 'Massimo', 'DeviazioneStandard', 'Conteggio']
all_plots_html_content = []

print("Generazione dei grafici per la pagina separata...")
for nome_parametro, df_grouped in grouped_data.items():
    if df_grouped is not None:
        for index, stats in df_grouped.iterrows():
            regione, anno = index

            plot_data = stats[statistiche_per_grafico].dropna()

            if not plot_data.empty:
                title = f"{nome_parametro} - {regione} ({anno}) - Tutte le Statistiche"
                y_label = f"Valore ({'giorni' if 'O3' in nome_parametro else 'µg/m^3'})"
                filename = f"{nome_parametro}_{regione.replace(' ', '_').replace('/', '_')}_{anno}_Statistiche.png"
                file_path = create_and_save_bar_plot(plot_data, title, y_label, filename)
                relative_image_path = os.path.join(PLOTS_DIR, filename).replace(os.sep, '/')
                
                #Aggiungi il contenuto HTML per questo grafico
                all_plots_html_content.append(f"""
                <div style="margin-bottom: 30px; padding: 15px; border: 1px solid #ddd; border-radius: 8px; background-color: #f9f9f9;">
                    <h3 style="color: #333; margin-top: 0;">{title}</h3>
                    <img src="{relative_image_path}" alt="{title}" style="max-width: 100%; height: auto; display: block; margin: 0 auto;">
                    <p style="font-size: 0.9em; color: #666; text-align: center;">Dati per {nome_parametro} in {regione} nell'anno {anno}.</p>
                </div>
                """)

print(f"Generati {len(all_plots_html_content)} grafici per la pagina separata.")

#Crea la pagina HTML con tutti i grafici
graphs_page_filename = 'Grafici In Dettaglio.html'
with open(graphs_page_filename, 'w', encoding='utf-8') as f:
    f.write(f"""
    <!DOCTYPE html>
    <html lang="it">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Dettaglio Grafici Qualità dell'Aria</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 20px; background-color: #f4f4f4; color: #333; }}
            .container {{ max-width: 900px; margin: 0 auto; padding: 20px; background-color: #fff; border-radius: 10px; box-shadow: 0 0 10px rgba(0,0,0,0.1); }}
            h1 {{ color: #0056b3; text-align: center; margin-bottom: 30px; }}
            .back-link {{ display: block; text-align: center; margin-bottom: 20px; }}
            .back-link a {{ text-decoration: none; color: #007bff; font-weight: bold; font-size: 1.1em; }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Grafici Dettaglio Qualità Dell'Aria Per Regione E Anno</h1>
            <div class="back-link">
                <a href="Mappa Qualità Aria Con Grafici.html">Torna alla Mappa Interattiva</a>
            </div>
            {"".join(all_plots_html_content)}
            <div class="back-link">
                <a href="Mappa Qualità Aria Con Grafici.html">Torna alla Mappa Interattiva</a>
            </div>
        </div>
    </body>
    </html>
    """)
print(f"Pagina di dettaglio grafici creata: {graphs_page_filename}")

Generazione dei grafici per la pagina separata...
Generati 1525 grafici per la pagina separata.
Pagina di dettaglio grafici creata: Grafici In Dettaglio.html


In [11]:
import folium
import io
import base64

regioni_coordinate = {
    'Abruzzo': (42.3500, 13.8500),
    'Basilicata': (40.6393, 15.8051),
    'Calabria': (38.9020, 16.5970),
    'Campania': (40.8395, 14.2545),
    'Emilia-Romagna': (44.5000, 11.0000),
    'Friuli-Venezia Giulia': (45.6494, 13.7780),
    'Lazio': (41.9000, 12.5000),
    'Liguria': (44.4000, 8.9300),
    'Lombardia': (45.6000, 9.3000),
    'Marche': (43.5000, 13.5000),
    'Molise': (41.5500, 14.6500),
    'Piemonte': (45.2000, 7.8000),
    'Puglia': (40.8000, 17.6000),
    'Sardegna': (40.0000, 9.0000),
    'Sicilia': (37.5000, 14.0000),
    'Toscana': (43.7700, 11.2500),
    'Trentino-Alto Adige': (46.5000, 11.0000),
    'Umbria': (43.1000, 12.5000),
    'Valle d\'Aosta': (45.7333, 7.3333),
    'Veneto': (45.6000, 11.8000)
}

def create_bar_plot_base64(data, title, y_label):
    fig, ax = plt.subplots(figsize=(6, 4))  
    colors = ['blue', 'orange', 'green', 'red', 'brown']
    bars = ax.bar(data.index, data.values, color=colors)
    ax.set_ylabel(y_label)
    ax.set_title(title)
    plt.xticks(rotation=45, ha='right', fontsize=8)  
    ax.yaxis.grid(True, linestyle='--', alpha=0.6)  
    ax.set_facecolor('lightgray')

    #Aggiungi tooltip
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3),
            textcoords="offset points",
            ha='center', va='bottom', fontsize=6)

    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    img_str = base64.b64encode(buf.getvalue()).decode('utf-8')
    plt.close(fig)
    return f'<img src="data:image/png;base64,{img_str}" alt="{title}">'

#Funzione per aggiungere marker con informazioni al popup
def aggiungi_marker_a_layer(layer, index, statistiche, nome_parametro, coordinate, grouped_data):
    regione = index[0]
    anno = index[1]
    if regione in coordinate:
        lat, lon = coordinate[regione]
        popup_content = f"<b>{regione} ({anno}) - {nome_parametro}</b><br>"
        for stat, valore in statistiche.items():
            if pd.isna(valore):
                popup_content += f"{stat}: N.D.<br>"
            else:
                popup_content += f"{stat}: {valore:.2f}<br>"

        popup_content += f"""
        <br>
        <a href="{graphs_page_filename}" target="_blank" style="text-decoration: none; color: #007bff; font-weight: bold;">
            Clicca per vedere tutti i grafici in una pagina separata
        </a>
        """

        folium.Marker([lat, lon], popup=folium.Popup(popup_content, max_width=600)).add_to(layer)

#Creazione della Mappa Folium
m = folium.Map(location=[41.8719, 12.5674], zoom_start=6)

o3_layer = folium.FeatureGroup(name='O3')
no2_layer = folium.FeatureGroup(name='NO2')
pm25_layer = folium.FeatureGroup(name='PM2.5')
pm10_layer = folium.FeatureGroup(name='PM10')

if grouped_o3_ra is not None:
    for index, stats in grouped_o3_ra.iterrows():
        aggiungi_marker_a_layer(o3_layer, index, stats.to_dict(), 'O3', regioni_coordinate, grouped_o3_ra)
o3_layer.add_to(m)

if grouped_no2_ra is not None:
    for index, stats in grouped_no2_ra.iterrows():
        aggiungi_marker_a_layer(no2_layer, index, stats.to_dict(), 'NO2', regioni_coordinate, grouped_no2_ra)
no2_layer.add_to(m)

if grouped_pm25_ra is not None:
    for index, stats in grouped_pm25_ra.iterrows():
        aggiungi_marker_a_layer(pm25_layer, index, stats.to_dict(), 'PM2.5', regioni_coordinate, grouped_pm25_ra)
pm25_layer.add_to(m)

if grouped_pm10_ra is not None:
    for index, stats in grouped_pm10_ra.iterrows():
        aggiungi_marker_a_layer(pm10_layer, index, stats.to_dict(), 'PM10', regioni_coordinate, grouped_pm10_ra)
pm10_layer.add_to(m)

folium.TileLayer('OpenStreetMap').add_to(m)
folium.LayerControl().add_to(m)

#Salva la mappa in un file HTML temporaneo
map_filename = 'Mappa Qualità Aria Con Grafici.html'
m.save(map_filename)

#Ora, apri il file HTML e inietta i link per il download dei grafici
with open(map_filename, 'r', encoding='utf-8') as f:
    html_content = f.read()

#Prepara il blocco HTML con il link alla pagina dei grafici
link_to_graphs_html = f"""
<div style="position: fixed; bottom: 10px; left: 10px; z-index: 1000; background-color: white; padding: 15px; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); max-height: 80vh; overflow-y: auto;">
    <h3 style="margin-top: 0; color: #333;">Esplora I Grafici In Dettaglio</h3>
    <p>
        <a href="{graphs_page_filename}" target="_blank" style="text-decoration: none; color: #007bff; font-weight: bold; font-size: 1.1em;">
            Visualizza tutti i grafici delle statistiche in una pagina separata
        </a>
    </p>
    <p style="font-size: 0.8em; color: #666;">
        Clicca sul link per aprire una nuova scheda con tutti i grafici generati.
    </p>
</div>
"""

#Trova il punto dove inserire il blocco HTML
insertion_point = '</body>'
html_content = html_content.replace(insertion_point, link_to_graphs_html + '\n' + insertion_point)

#Salva il file HTML
with open(map_filename, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"Mappa interattiva con link alla pagina dei grafici creata in: {map_filename}")

Mappa interattiva con link alla pagina dei grafici creata in: Mappa Qualità Aria Con Grafici.html
